In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
from astropy.io import fits
from spectractor.simulation.throughput import TelescopeTransmission
from spectractor.config import load_config
from spectractor.extractor.targets import Star
from spectractor.extractor.dispersers import Hologram
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from scipy.integrate import trapezoid

%matplotlib widget

In [ ]:
data_dir = '/home/lmousset/Projets_Recherche/LSST/Star_catalogue/'

load_config("auxtel.ini")

In [ ]:
def load_dataframe_filtered(data_dir):
    # Load the dataframe
    df = pd.read_parquet(data_dir + "auxtel_atmosphere_202311_v3_2_1_fixA2fixA1_RobustFit_newThroughputs.gz")

    # Apply some cuts to select good quality spectra
    print(len(df))
    filtered = (df["CHI2_FIT"] < 200) 
    print(np.sum(filtered))
    filtered = filtered  & (df["PSF_REG"] > 1e-2) 
    print(np.sum(filtered))
    filtered = filtered  & (df["D2CCD"] > 186.0)  & (df["D2CCD"] < 188)  
    print(np.sum(filtered))
    filtered = filtered  & (df["PIXSHIFT"] > -1.)  & (df["PIXSHIFT"] < 1) 
    print(np.sum(filtered))
    filtered = filtered  & (df["alpha_0_2"] < 6)  & (df["alpha_0_1"] >1.2)  & (df["gamma_0_1"] < 20) # & (rec["gamma_0_2"] < 20)
    print(np.sum(filtered))
    filtered = filtered  & (df["PWV [mm]_err_rum"] > 0) & (df["PWV [mm]_err"] > 0) # & (rec["PWV [mm]_err"] < 5)
    print(np.sum(filtered))


    df_filtered = df[filtered]
    #print(df_filtered["id"])

    # Add a column for the night of observation
    df_filtered["night"] = (df_filtered["id"] // 100_000).astype(int)

    return df_filtered

def plot_spectra(specs, colorparams, title=None):
    colormap = cm.Reds
    normalize = mcolors.Normalize(vmin=np.min(colorparams), vmax=np.max(colorparams))

    fig  = plt.figure(figsize=(7,3))
    for s, spec in enumerate(specs):
        plt.plot(spec[0, :], spec[1, :], color = colormap(normalize(colorparams[s])))
    plt.grid()
    plt.xlabel(r"$\lambda$ [nm]")
    plt.ylabel(f"Flux ")
    if title:
        plt.title(title)

    # Colorbar setup
    s_map = cm.ScalarMappable(norm=normalize, cmap=colormap)
    s_map.set_array(colorparams)

    # Use this to emphasize the discrete color values
    cbar = fig.colorbar(s_map, ax=plt.gca())

    cbar.set_label(r"Airmass $z$")
    plt.tight_layout()
    return fig

def make_freq_sampling(filter, step=1):
    if filter=="OG550_65mm_1":
        lambda_min, lambda_max = 550, 1025  #1050
    elif filter=="BG40_65mm_1":
        lambda_min, lambda_max = 370, 650
    elif filter=="empty":
        lambda_min, lambda_max = 370, 1025 #1050
    else:
        raise ValueError(f"Unknown filter {filter}")

    wls = np.arange(lambda_min, lambda_max, step) # Wavelengths of interest
    return wls

def make_spec_toa(pvals, pcovs):
    spec_toa = 10**(-0.4 * pvals[:, 1])
    spec_toa_err = np.abs(-0.4 * np.log(10) * spec_toa * np.sqrt(pcovs[:, 1, 1]))
    return spec_toa, spec_toa_err

def make_tel_transmission_factor(wls, filter="empty"):

    # Load the disperser
    h = Hologram("holo4_003")

    # Detector gain
    gain = 0.8

    tel = TelescopeTransmission(filter_label=filter)
    
    factor = tel.transmission(wls) * gain * h.transmission(wls)
    return factor

def make_reference_sed(target, wls):
    s = Star(target)
    sed = s.sed(wls)
    return sed


def polyfit_sigmaclip(x, y, yerr, deg=1, sigmaclip=5, niter=3):
    w = 1/yerr
    goods = (~np.isnan(y)) & (~np.isnan(yerr)) & (yerr > 0)  # Erreurs > 0
    w[~goods] = 0
    
    # Vérifier qu'on a assez de points
    if np.sum(w > 0) <= deg + 1:
        print(f"Warning: Only {np.sum(w > 0)} points, but need {deg + 1}")
        return np.zeros(deg + 1), np.zeros((deg + 1, deg + 1)), (w == 0.)
    
    # Régulariser les poids trop petits/grands
    valid_w = w[w > 0]
    w_min = np.min(valid_w)
    w_max = np.max(valid_w)
    if w_max / w_min > 1e6:  # Trop grande disparité
        w = np.clip(w, w_min * 100, w_max / 100)
    
    # Weighted fit with error handling
    for n in range(niter):
        try:
            pval, pcov = np.polyfit(x[goods], y[goods], w=w[goods], deg=deg, cov='unscaled')
        except np.linalg.LinAlgError:
            print(f"Singular matrix at iteration {n}, returning zeros")
            return np.zeros(deg + 1), np.zeros((deg + 1, deg + 1)), (w == 0.)
        
        model = np.polyval(pval, x)
        res = np.abs((y - model) * w)
        clip = res > sigmaclip
        
        if np.any(clip):
            worst = np.argmax(res)
            w[worst] = 0
            if np.sum(w > 0) <= deg + 1:  # Vérifier après clipping
                break
        else:
            break
    
    return pval, pcov, (w == 0.)


def get_files_pvals_pcovs(spectra_dir, night_min, night_max, filt, target):
    # Récupérer tous les fichiers pvals et pcovs pour cette étoile et ce filtre
    all_nights = []
    all_pvals_list = []
    all_pcovs_list = []

    # Parcourir tous les répertoires de nuit
    for night_folder in sorted(os.listdir(spectra_dir)):
        night_path = os.path.join(spectra_dir, night_folder)
        if not os.path.isdir(night_path):
            continue
        
        try:
            night = int(night_folder)
            if night < night_min or night > night_max:
                continue
        except ValueError:
            continue
        
        # Vérifier si le filtre existe pour cette nuit
        filt_path = os.path.join(night_path, filt)
        if not os.path.isdir(filt_path):
            continue
        
        # Vérifier si l'étoile existe pour ce filtre
        star_path = os.path.join(filt_path, target)
        if not os.path.isdir(star_path):
            continue
        
        # Charger les fichiers pvals et pcovs
        pvals_file = os.path.join(star_path, "pvals.npy")
        pcovs_file = os.path.join(star_path, "pcovs.npy")
        
        if os.path.exists(pvals_file) and os.path.exists(pcovs_file):
            pvals = np.load(pvals_file)
            pcovs = np.load(pcovs_file)
            #print(f"Night {night}: pvals shape {pvals.shape}, pcovs shape {pcovs.shape}")
            
            all_nights.append(night)
            all_pvals_list.append(pvals)
            all_pcovs_list.append(pcovs)

    print(f"\nTotal nights found: {len(all_nights)}")
    print(f"Nights: {all_nights}")

    return all_nights, all_pvals_list, all_pcovs_list

In [ ]:
# Load the dataframe
df_filtered = load_dataframe_filtered(data_dir)
df_filtered.head()

In [ ]:
all_targets = np.unique(df_filtered["TARGET"]) # All stars that were observed
Ntargets = len(all_targets)
print(all_targets, Ntargets)

In [ ]:
all_nights = np.unique(df_filtered["night"]) # All nights

Nnights = all_nights.size
print(Nnights)
print(all_nights.min(), all_nights.max())

# Plot the spectra for one target and one filter

In [ ]:
# Choose a target and a filter and the night range to explore
mytarget = "HD2811"
myfilter = "empty"

night_min = 20251107
night_max = 20251108

In [ ]:
# Loop over nights
for night, groupn in df_filtered.groupby("night"):
    if night < night_min:
        continue
    if night > night_max:
        break
    # Loop over filters
    for filt, groupf in groupn.groupby("FILTER"):
        if filt == myfilter:
            # Loop over targets
            for target, groupt in groupf.groupby(["TARGET"]):
                if target[0] == mytarget:
                    # Number of spectra for this target
                    nspecs = len(groupt["id"]) 
                    print(f"Target {target} with filter {filt} has {nspecs} spectra")
                    
                    # Get spectra and airmasses
                    specs = [np.load(data_dir + f"Saved_spectra/{night}/{filt}/{target[0]}/spec_visit{i:02d}.npy") for i in range(nspecs)]
                    airmasses = [groupt.iloc[i]["AIRMASS"] for i in range(nspecs)]
                    print(f'Airmasses: {airmasses}')
                    
                    # Plot
                    fig = plot_spectra(specs, airmasses)
                    fig.suptitle(f"Night {night} - Target {target[0]} with filter {filt}")

In [ ]:
# Define the wavelength range
wls_sed = make_freq_sampling(myfilter, step=1)

colormap = cm.OrRd_r
normalize = mcolors.Normalize(vmin=0, vmax=Ntargets)


# Plot the SEDs of all targets
fig = plt.figure(figsize=(12, 5))
for i, target in enumerate(all_targets):
    s = Star(target)
    sed = s.sed(wls_sed)
    plt.plot(wls_sed, sed, label=target, color=colormap(normalize(i)))
plt.yscale("log")
plt.xlabel("Wavelength (nm)")
plt.ylabel("Reference SED")
plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize='small')

# Analysis

Save pvals and pcovs files for each star, each filter, each night.

In [ ]:
# Choose a target and a filter and the night range to explore
save = True

night_min = 20231107
night_max = 20251231

sigmaclip_cloud = 10
niter_cloud = 10

sigmaclip = 10
niter = 10

step_wls = 1

# Loop over nights
for night, groupn in df_filtered.groupby("night"):
    print(f"\n============== Night {night} ==============")
    if night < night_min:
        continue
    if night > night_max:
        break
    # Loop over filters
    for filt, groupf in groupn.groupby("FILTER"):
        print(f"Filter {filt}")
        wls = make_freq_sampling(filt, step=step_wls) # Wavelengths of interest
        nwls = len(wls)
                
        # Loop over targets
        for target, groupt in groupf.groupby(["TARGET"]):
            # Number of spectra for this target
            nspecs = len(groupt["id"]) 
            print(f"Target {target} with filter {filt} has {nspecs} spectra")
            
            ### Get spectra and airmasses
            all_specs = []
            all_airmasses = np.zeros(nspecs)
            all_amplitudes = np.zeros(nspecs)
            all_mags = np.zeros((nspecs, nwls))
            all_dmags = np.zeros((nspecs, nwls))
            # Loop over spectra            
            for i in range(nspecs):
                spec = np.load(data_dir + f"Saved_spectra/{night}/{filt}/{target[0]}/spec_visit{i:02d}.npy")
                airmass = groupt.iloc[i]["AIRMASS"]
                amplitude = -2.5*np.log10(np.sum(spec[1, :]))
                all_airmasses[i] = airmass
                all_amplitudes[i] = amplitude
                all_specs.append(spec)

                ### Compute magnitudes and uncertainties on magnitudes at the wavelengths of interest
                mags = np.zeros(nwls) # magnitudes
                dmags = np.zeros(nwls) # uncertainties on magnitudes
                for l, wl in enumerate(wls):
                    mags[l] = -2.5*np.log10(np.interp(wl, spec[0, :], spec[1, :]))
                    dmags[l] = 2.5/np.log(10) * np.abs(np.interp(wl, 
                                                                 spec[0, :], 
                                                                 spec[2, :])/np.interp(wl, spec[0, :], spec[1, :])
                                                                 )
                all_mags[i, :] = mags
                all_dmags[i, :] = dmags
            
            #plot_spectra(all_specs, all_airmasses)
            
            
            if nspecs > 2:
                #### Cut the cloud presence looking at spectrum integral
                pval, pcov, clipped = polyfit_sigmaclip(np.array(all_airmasses), np.array(all_amplitudes), deg=1,
                                                        yerr=5e-5*all_amplitudes, 
                                                        sigmaclip=sigmaclip_cloud, 
                                                        niter=niter_cloud)
            
                clipped = np.array(clipped, dtype=bool)
                #plot_spectra([spec for i, spec in enumerate(all_specs) if clipped[i] == False], 
                 #            all_airmasses[~clipped],
                  #           title='Clipped')
                print(f"Clipped spectra: {np.sum(clipped)} out of {nspecs}")
                # Set the uncertainties on magnitudes to infinity for the clipped spectra
                all_dmags[clipped, :] = np.inf

                if nspecs - np.sum(clipped) <= 2:
                    print("Not enough spectra left after clipping to perform airmass regression")
                    continue
                else:
                    ### Airmass regression
                    #fig, axs = plt.subplots(1, 2, figsize=(12, 5))
                    #axs = axs.flatten()
                    all_pvals = np.zeros((nwls, 2))
                    all_pcovs = np.zeros((nwls, 2, 2))
                    for k, wl in enumerate(wls):
                        pval, pcov, clipped = polyfit_sigmaclip(all_airmasses, all_mags[:, k], deg=1, yerr=all_dmags[:, k],
                                                                sigmaclip=sigmaclip, niter=niter)
                        
                        all_pvals[k, :] = pval
                        all_pcovs[k, :, :] = pcov

                        #p = axs[0].errorbar(all_airmasses[~clipped], all_mags[:, k][~clipped], yerr=all_dmags[:, k][~clipped], marker='+', linestyle='none', label=wl)
                        #axs[0].plot(all_airmasses, np.polyval(pval, all_airmasses), color=p[0].get_color())

                        #residus = all_mags[:, k] - np.polyval(all_pvals[k], all_airmasses)
                        #p = axs[1].errorbar(all_airmasses[~clipped], residus[~clipped], yerr=all_dmags[:, k][~clipped], marker='+', linestyle='none', label=wl)
                    
                    #axs[0].grid()
                    #axs[0].legend()
                    #axs[0].set_ylabel("Magnitude [mag]")
                    #axs[0].set_xlabel("Airmass")

                    #axs[1].grid()
                    #axs[1].set_xlabel("Airmass")
                    #axs[1].set_ylabel("Residuals to airmass regression [mag]")
                    #axs[1].legend()

                    if save:
                        np.save(data_dir + f"Saved_spectra/{night}/{filt}/{target[0]}/pvals.npy", all_pvals)
                        np.save(data_dir + f"Saved_spectra/{night}/{filt}/{target[0]}/pcovs.npy", all_pcovs)


# Look at the regression parameters

In [ ]:
# Load the regression results and plot the airmass regression coefficients as a function of wavelength
night = 20251115
filt = "OG550_65mm_1"
mytarget = "HD14943"

wls = make_freq_sampling(filt, step=1) # Wavelengths of interest
wls_sed = make_freq_sampling(filt, step=1) # All wavelengths for the SED plots

pvals = np.load(data_dir + f"Saved_spectra/{night}/{filt}/{mytarget}/pvals.npy")
pcovs = np.load(data_dir + f"Saved_spectra/{night}/{filt}/{mytarget}/pcovs.npy")
print(pvals.shape, pcovs.shape)

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs = axs.flatten()
fig.suptitle(f"Airmass regression coefficients for target {mytarget} with filter {filt} on night {night}")

axs[0].errorbar(wls, pvals[:, 0], yerr=np.sqrt(pcovs[:, 0, 0]), marker='o', linestyle='none')
axs[0].set_xlabel("Wavelength [nm]")
axs[0].set_ylabel("Slope [mag]")
axs[0].grid()

axs[1].errorbar(wls, pvals[:, 1], yerr=np.sqrt(pcovs[:, 1, 1]), marker='o', linestyle='none')
axs[1].set_xlabel("Wavelength [nm]")
axs[1].set_ylabel("Intercept [mag]")
axs[1].grid()

In [ ]:
### Star spectrum without the atmosphere (but with the telescope transmission)
spec_toa, spec_toa_err = make_spec_toa(pvals, pcovs)

transmission_factor = make_tel_transmission_factor(wls_sed, filter=filt)

sed = make_reference_sed(mytarget, wls_sed)

# SED with telescope transmission
sed_withtel = sed * transmission_factor
sed_normalized = sed_withtel * trapezoid(spec_toa, x=wls) / trapezoid(sed_withtel, x=wls_sed)

# Spectrum without tel transmission
spec_toa_notel = spec_toa / transmission_factor
spec_toa_notel_err = spec_toa_err / transmission_factor

# SED without telescope transmission 
sed_notel_normalized = sed * trapezoid(y=spec_toa_notel, x=wls) / trapezoid(y=sed, x=wls_sed)

# Plot
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs = axs.flatten()
fig.suptitle(f"Target {mytarget} - Filter {filt} - Night {night}")

axs[0].errorbar(wls, spec_toa, yerr=spec_toa_err, marker="+", color="k", linestyle="none")
axs[0].plot(wls_sed, sed_normalized)
axs[0].set_xlabel(r"$\lambda$ [nm]")
axs[0].set_ylabel(f"Spectrum TOA with throughput")
axs[0].set_title(f"With telescope transmission")
axs[0].grid()

axs[1].errorbar(wls, spec_toa_notel, yerr=spec_toa_notel_err, marker="+", color="k", linestyle="-")
#axs[1].plot(wls_sed, sed_notel_normalized)
axs[1].set_ylim(0, 1.2*np.max(spec_toa_notel))
axs[1].set_xlabel(r"$\lambda$ [nm]")
axs[1].set_ylabel(f"Spectrum TOA")
axs[1].set_title(f"Without telescope transmission")
axs[1].grid()


# Mean of regression parameters for one star

In [ ]:
print(all_targets)

In [ ]:
night_min = 20231107
night_max = 20251231

#filt= "empty"
#filt = "BG40_65mm_1"
filt ="OG550_65mm_1"

wls = make_freq_sampling(filt, step=1) # Wavelengths of interest
wls_sed = make_freq_sampling(filt, step=1) # Wavelengths of interest

transmission_factor = make_tel_transmission_factor(wls, filter=filt)

colormap = cm.get_cmap('viridis')  # Bonne colormap

all_ratio = []
all_target_observed = []
for target in all_targets:
    print(f"Target {target}")
    
    all_nights, all_pvals_list, all_pcovs_list = get_files_pvals_pcovs(data_dir + "Saved_spectra", night_min, night_max, filt, target)

    if len(all_nights) == 0:
        print(f"No data found for target {target} with filter {filt} in the specified night range.")
        continue

    elif target=="HD185975":
        print('Etoile moche, on skip')

    else:
        all_target_observed.append(target)

        # Median spectrum TOA over all nights
        median_pvals = np.median(np.array(all_pvals_list), axis=0)
        median_pcovs = np.median(np.array(all_pcovs_list), axis=0)
        median_spec_toa, median_spec_toa_err = make_spec_toa(median_pvals, median_pcovs)

        # Mean spectrum TOA over all nights
        mean_pvals = np.mean(np.array(all_pvals_list), axis=0)
        mean_pcovs = np.mean(np.array(all_pcovs_list), axis=0)
        mean_spec_toa2, mean_spec_toa_err2 = make_spec_toa(mean_pvals, mean_pcovs)

        # Mean spectrum TOA over all nights
        mean_spec_toa = np.zeros(len(wls))
        mean_spec_toa_err = np.zeros(len(wls))
        for pvals, pcovs in zip(all_pvals_list, all_pcovs_list):
            spec_toa, spec_toa_err = make_spec_toa(pvals, pcovs)
            mean_spec_toa += spec_toa
            mean_spec_toa_err += spec_toa_err**2
        mean_spec_toa /= len(all_pvals_list)
        mean_spec_toa_err = np.sqrt(mean_spec_toa_err) / len(all_pvals_list)

        ### SED of the target
        sed = make_reference_sed(target, wls_sed)
        sed_trans = sed * transmission_factor
        sed_trans_normalized = sed_trans*trapezoid(y=median_spec_toa, x=wls) / trapezoid(y=sed_trans, x=wls_sed)
        


        #### Star spectrum without the atmosphere (but with the telescope transmission)
        fig = plt.figure(figsize=(12, 7))
        normalize = mcolors.Normalize(vmin=0, vmax=len(all_pvals_list)-1)

        for i, (pvals, pcovs) in enumerate(zip(all_pvals_list, all_pcovs_list)):
            #print(f"Night {all_nights[i]}: pvals shape {pvals.shape}, pcovs shape {pcovs.shape}")
            spec_toa, spec_toa_err = make_spec_toa(pvals, pcovs)
            
            
            
            color = colormap(normalize(i))
            plt.errorbar(wls, spec_toa, yerr=spec_toa_err, marker=".", linestyle="-", 
                        label=f'{all_nights[i]}', color=color, alpha=0.5)
        
        plt.errorbar(wls, mean_spec_toa, yerr=mean_spec_toa_err, marker=".", linestyle="-", color="purple", label="Mean spectrum")
        plt.errorbar(wls, mean_spec_toa2, yerr=mean_spec_toa_err2, marker=".", linestyle="-", color="orange", label="Mean spectrum 2")
        plt.errorbar(wls, median_spec_toa, yerr=median_spec_toa_err, marker=".", linestyle="-", color="red", label="Median spectrum")

        plt.plot(wls_sed, sed_trans_normalized, 'k', label='ReferenceSED', zorder=42)
        plt.xlabel(r"$\lambda$ [nm]")
        plt.ylabel(f"Spectrum TOA with throughput")
        plt.title(f"{target} - With telescope transmission")
        try:
            plt.ylim(0, 4*np.mean(median_spec_toa))
        except:
            pass
        plt.grid()
        plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize='small')
        plt.tight_layout()
        plt.show()

        ### Star spectrum without the atmosphere and without the telescope transmission
        mean_spec_toa /= transmission_factor
        mean_spec_toa_err /= transmission_factor
        mean_spec_toa2 /= transmission_factor
        mean_spec_toa_err2 /= transmission_factor
        median_spec_toa /= transmission_factor
        median_spec_toa_err /= transmission_factor
        sed_normalized = sed*trapezoid(y=median_spec_toa, x=wls) / trapezoid(y=sed, x=wls_sed)

        fig = plt.figure(figsize=(12, 7))
        normalize = mcolors.Normalize(vmin=0, vmax=len(all_pvals_list)-1)

        for i, (pvals, pcovs) in enumerate(zip(all_pvals_list, all_pcovs_list)):
            #print(f"Night {all_nights[i]}: pvals shape {pvals.shape}, pcovs shape {pcovs.shape}")
            spec_toa, spec_toa_err = make_spec_toa(pvals, pcovs)
            spec_toa /= transmission_factor
            spec_toa_err /= transmission_factor
            
            color = colormap(normalize(i))
            plt.errorbar(wls, spec_toa, yerr=spec_toa_err, marker=".", linestyle="-", 
                        label=f'{all_nights[i]}', color=color, alpha=0.5)
        
        plt.errorbar(wls, mean_spec_toa, yerr=mean_spec_toa_err, marker=".", linestyle="-", color="purple", label="Mean spectrum")
        plt.errorbar(wls, mean_spec_toa2, yerr=mean_spec_toa_err2, marker=".", linestyle="-", color="orange", label="Mean spectrum 2")
        plt.errorbar(wls, median_spec_toa, yerr=median_spec_toa_err, marker=".", linestyle="-", color="red", label="Median spectrum")

        plt.plot(wls_sed, sed_normalized, 'k', label='ReferenceSED', zorder=42)
        plt.xlabel(r"$\lambda$ [nm]")
        plt.ylabel(f"Spectrum TOA with throughput")
        plt.title(f"{target} - No telescope transmission")
        try:
            plt.ylim(0, 4*np.mean(median_spec_toa))
        except:
            pass
        plt.grid()
        plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize='small')
        plt.tight_layout()
        plt.show()

        ### Ratio
        sed_normalized_at_wls = np.interp(wls, wls_sed, sed_normalized)
        ratio = median_spec_toa / sed_normalized_at_wls
        all_ratio.append(ratio)



In [ ]:
plt.figure(figsize=(12, 7))
for i in range(len(all_ratio)):
     plt.plot(wls, all_ratio[i], marker=".", linestyle="-", label=f"{all_target_observed[i]}", alpha=0.7)
    
plt.xlabel(r"$\lambda$ [nm]")
plt.ylabel(f"Spectrum TOA / Reference SED")
plt.title(f"{target} - Filter {filt} - Median spectrum / Reference SED")
plt.grid()
plt.ylim(0.5, 1.5)
plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize='small')
plt.tight_layout()
plt.show()